# Advanced YOLOv8 Colab Training Pipeline (Crash-Proof)

This notebook is specifically designed for Google Colab. It mounts your Google Drive immediately and saves all checkpoints directly to the Drive. If Colab disconnects, simply reconnect and click 'Run All'—the training will automatically resume from the exact epoch it crashed on.

In [3]:
# 1. Install Dependencies & Mount Google Drive
!pip install ultralytics roboflow -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 2. Download Dataset from Roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="rkk8UgicFLnBv2uW8xrc")
project = rf.workspace("nareks-workspace-gfmp5").project("military-vehicles-detection-qwfnc-8qvf8")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Military-vehicles-detection-1 in yolov8:: 100%|██████████| 19929/19929 [00:09<00:00, 2126.99it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
# 3. Clean Labels (Remove Civilian Cars)
import os

DATASET_DIR = "/content/Military-vehicles-detection-1"
CLASS_TO_REMOVE = 1

def clean_labels(folder_name):
    labels_path = os.path.join(DATASET_DIR, folder_name, 'labels')
    if not os.path.exists(labels_path): return

    for filename in os.listdir(labels_path):
        if not filename.endswith('.txt'): continue
        file_path = os.path.join(labels_path, filename)

        with open(file_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts: continue
            class_id = int(parts[0])
            if class_id == CLASS_TO_REMOVE: continue
            if class_id > CLASS_TO_REMOVE: class_id -= 1
            parts[0] = str(class_id)
            new_lines.append(" ".join(parts) + "\n")

        with open(file_path, 'w') as f:
            f.writelines(new_lines)

for folder in ['train', 'valid', 'test']:
    clean_labels(folder)
print("Dataset cleaned: Civilian Cars removed.")

import yaml
yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)
if isinstance(data.get('names'), list) and "CIVILIAN_CAR" in data['names']:
    data['names'].remove("CIVILIAN_CAR")
    data['nc'] = len(data['names'])
elif isinstance(data.get('names'), dict) and 1 in data['names']:
    del data['names'][1]
    new_names = {i: name for i, name in enumerate(data['names'].values())}
    data['names'] = new_names
    data['nc'] = len(data['names'])
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)
print("data.yaml successfully updated!")

Dataset cleaned: Civilian Cars removed.
data.yaml successfully updated!


In [6]:
# 4. Train with Crash-Proof Saving
from ultralytics import YOLO
import os

# Check if a previous run crashed so we can resume it
resume_weights = "/content/drive/MyDrive/military_yolo_research/yolov8m_heavy_aug/weights/last.pt"
if os.path.exists(resume_weights):
    print("Found interrupted training! Resuming from Google Drive checkpoint...")
    model = YOLO(resume_weights)
    resume_flag = True
else:
    print("Starting fresh training...")
    model = YOLO("yolov8m.pt")
    resume_flag = False

results = model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=50,
    patience=25,
    save=True,
    save_period=10,
    imgsz=640,
    batch=16,
    device=0,
    resume=resume_flag, # Automatically picks up from last.pt if it exists

    # --- CRASH PROOF DIRECTORY ---
    project="/content/drive/MyDrive/military_yolo_research", # SAVES DIRECTLY TO GOOGLE DRIVE
    name="yolov8m_heavy_aug",

    # --- HEAVY DATA AUGMENTATION PARAMETERS ---
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.2, scale=0.6,
    perspective=0.0005, fliplr=0.5, mosaic=1.0,
    mixup=0.2, erasing=0.4, copy_paste=0.2
)
print("Training Complete! Weights are already in your Google Drive.")

Starting fresh training...
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Military-vehicles-detection-1/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_heavy_aug, nbs=64, nms=False, opset=None, opt